In [1]:
from lmabo import LanguageModelAssistedAdaptiveBO
from baselines.bo_helpers import bo_single_iteration
from utils import get_shortest_distance_from_last_point
from lmabo import FOLLOW_UP_PROMPT_TEMPLATE_LIST

import numpy as np

class LMABOWithModifiedPrompt(LanguageModelAssistedAdaptiveBO):
    def optimize_single_step(self):
        # use LLM to suggest the best acq_type
        acq_type = self.convo.suggest_acq_type(self._construct_prompt())
        if acq_type == "Intentional Incorrect AF":
            exit()
        self.acq_type_list.append(acq_type)
        # run one BO iter with the acq_type suggested by LLM
        self.train_X, self.train_Y, self.gp = bo_single_iteration(
            self.train_X, 
            self.train_Y, 
            acq_type, 
            self.objective_func, 
            self.bounds
        )
        self.lengthscales = self.gp.covar_module.base_kernel.lengthscale.detach().cpu().numpy()
        self.outputscale = self.gp.covar_module.outputscale.detach().cpu().numpy()
        # Store best observed value
        self.best_values.append(self.train_Y.min().item())
        print(f"Current best value: {self.train_Y.min().item()}")
        self.remaining_iterations -= 1

    def get_next_prompt_elements(self):
        # --- NEW: Calculate shortest distance of the last point relative to bounds ---
        shortest_dist = get_shortest_distance_from_last_point(self.train_X, self.bounds)
        # --- Calculate descriptive statistics ---
        min_ls = np.min(self.lengthscales)
        max_ls = np.max(self.lengthscales)
        mean_ls = np.mean(self.lengthscales)
        std_ls = np.std(self.lengthscales)

        prompt_elements = {
            "N": self.train_Y.shape[0],
            "remaining": self.remaining_iterations,
            "D": self.train_X.shape[1],
            "f_max": np.round(self.train_Y.max().detach().cpu().numpy(), decimals=3).item(),
            "f_mean": np.round(self.train_Y.mean().detach().cpu().numpy(), decimals=3).item(),
            "f_std": np.round(self.train_Y.std().detach().cpu().numpy(), decimals=3).item(),
            "f_min": np.round(self.train_Y.min().detach().cpu().numpy(), decimals=3).item(),
            "shortest_dist": shortest_dist,
            "min_ls": min_ls,
            "max_ls": max_ls,
            "mean_ls": mean_ls,
            "std_ls": std_ls,
            "outputscale": self.outputscale
        }
        return prompt_elements

/home/s222509501/.conda/envs/lmabo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from run import (
    setup_experiment,
    generate_initial_data,
)

    # if element == "remaining":
    #     delta_list = [-50, -20, -10, -5, -1, 1, 5, 10, 20, 50]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["remaining"] = modified_elements["remaining"] + delta
    #         # Make sure remaining is always positive, if not just continue
    #         if modified_elements["remaining"] <= 0:
    #             continue
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["remaining"])
    # elif element == "f_max":
    #     delta_list = [0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         f_min, f_mean = modified_elements["f_min"], modified_elements["f_mean"]
    #         # make sure f_max is always larger than f_mean and f_min and multiplied by delta, if not just continue
    #         new_f_max = round(modified_elements["f_max"] * delta, 3)
    #         if new_f_max <= f_mean or new_f_max <= f_min:
    #             continue
    #         modified_elements["f_max"] = new_f_max
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["f_max"])
    # elif element == "f_mean":
    #     delta_list = [0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["f_mean"] = round(modified_elements["f_mean"] * delta, 3)
    #         # make sure f_mean is always between f_max and f_min, if not just continue
    #         if modified_elements["f_mean"] >= modified_elements["f_max"] or modified_elements["f_mean"] <= modified_elements["f_min"]:
    #             continue
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["f_mean"])
    # elif element == "f_std":
    #     delta_list = [0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["f_std"] = max(0.001, round(modified_elements["f_std"] * delta, 3))
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["f_std"])
    # elif element == "shortest_dist":
    #     delta_list = [0.01, 0.1, 0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5, 2, 5, 10, 100]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["shortest_dist"] = round(modified_elements["shortest_dist"] * delta, 3)
    #         # Make sure it is not larger than sqrt of dimension, if not just continue
    #         if modified_elements["shortest_dist"] > np.sqrt(modified_elements["D"]):
    #             continue
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["shortest_dist"])
    # elif element == "min_ls":
    #     delta_list = [0.01, 0.1, 0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5, 2, 5, 10]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["min_ls"] = round(modified_elements["min_ls"] * delta, 3)
    #         # Make sure it is not larger than mean_ls, if not just continue
    #         if modified_elements["min_ls"] > modified_elements["mean_ls"]:
    #             continue
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["min_ls"])
    # elif element == "max_ls":
    #     delta_list = [0.01, 0.1, 0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5, 2, 5, 10, 100]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["max_ls"] = max(0.001, round(modified_elements["max_ls"] * delta, 3))
    #         # Make sure it is not smaller than mean_ls, if not just continue
    #         if modified_elements["max_ls"] < modified_elements["mean_ls"]:
    #             continue
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["max_ls"])
    # elif element == "mean_ls":
    #     delta_list = [0.01, 0.1, 0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5, 2, 5, 10, 100]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["mean_ls"] = max(0.001, round(modified_elements["mean_ls"] * delta, 3))
    #         # Make sure it is between min_ls and max_ls, if not just continue
    #         if modified_elements["mean_ls"] < modified_elements["min_ls"] or modified_elements["mean_ls"] > modified_elements["max_ls"]:
    #             continue
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["mean_ls"])
    # elif element == "std_ls":
    #     delta_list = [0.01, 0.1, 0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5, 2, 5, 10, 100]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["std_ls"] = max(0.001, round(modified_elements["std_ls"] * delta, 3))
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["std_ls"])
    # elif element == "outputscale":
    #     delta_list = [0.01, 0.1, 0.5, 0.75, 0.9, 0.99, 1.01, 1.1, 1.25, 1.5, 2, 5, 10, 100]
    #     for delta in delta_list:
    #         modified_elements = prompt_elements.copy()
    #         modified_elements["outputscale"] = max(0.001, round(modified_elements["outputscale"] * delta, 3))
    #         modified_prompts.append(assemble_prompt(modified_elements))
    #         all_modified_elements.append(modified_elements["outputscale"])

def assemble_prompt(prompt_elements):
    prompt = FOLLOW_UP_PROMPT_TEMPLATE_LIST[0].format(
        N=prompt_elements["N"],
        remaining=prompt_elements["remaining"],
        D=prompt_elements["D"],
        f_max=prompt_elements["f_max"],
        f_mean=prompt_elements["f_mean"],
        f_std=prompt_elements["f_std"],
        f_min=prompt_elements["f_min"],
        shortest_dist=prompt_elements["shortest_dist"],
        min_ls=prompt_elements["min_ls"],
        max_ls=prompt_elements["max_ls"],
        mean_ls=prompt_elements["mean_ls"],
        std_ls=prompt_elements["std_ls"],
        outputscale=prompt_elements["outputscale"]
    )
    return prompt

def sensitivity_analysis(problem, element, element_val, starting_iterations, seed):
    objective_func, bounds, num_initial_points, num_iterations = setup_experiment(problem)
    fixed_train_X, fixed_train_Y  = generate_initial_data(
        bounds, 
        num_initial_points, 
        seed, 
        objective_func
    )
    # Run the optimization for N steps
    LMABO = LMABOWithModifiedPrompt(
        objective_func, 
        fixed_train_X, 
        fixed_train_Y, 
        bounds, 
        num_iterations,
        "api",
        "localhost",
    )
    for _ in range(starting_iterations):
        LMABO.optimize_single_step()
    # Get the next prompt elements
    prompt_elements = LMABO.get_next_prompt_elements()
    # Modify the prompt by changing the current concerned element
    prompt_elements[element] = element_val
    # Get the next acq_type given modified prompts
    modified_acq_types = []
    mod_acq_type = LMABO.convo.suggest_acq_type(assemble_prompt(prompt_elements))
    modified_acq_types.append(mod_acq_type)
    print(f"Modified element value: {element_val} | Suggested acq_type: {mod_acq_type}")
    # Get original acq_type
    original_prompt = assemble_prompt(prompt_elements)
    original_acq_type = LMABO.convo.suggest_acq_type(original_prompt)
    return original_acq_type, modified_acq_types, original_prompt

In [6]:
original_acq_type, modified_acq_types, original_prompt, all_modified_elements = sensitivity_analysis(
    problem="Griewank",
    element="remaining",
    starting_iterations=5,
    seed=42
)

Using valid key: AIzaSyCs...
Starting Gemini chat session with initial context...
Gemini's initial acknowledgement: Yes, I understand the task and the available acquisition functions. I am ready to receive the optimization summary and recommend the most suitable acquisition function for the next iteration, adhering to the specified format and constraints.
Iter 0| 
    Current optimization state:
    - N: 19 
    - Remaining iterations: 50
    - D: 9
    - f_range: Range [156.444, 413.750], Mean 269.369 (Std Dev 78.245)
    - f_min: 156.444
    - Shortest distance: 0.8864098106485899
    - Lengthscales: Range [0.275, 2243.573], Mean 857.552 (Std Dev 783.602)
    - Outputscale: 1.026403025520932
    
Total tokens used so far:  0
Estimated cost so far: $0.000000
LLM suggested AF: qMES justified by: Given the high dimensionality (D=9) and the significant variability in model lengthscales, qMES is ideal for efficiently reducing uncertainty about the location of the global optimum, which is 

In [ ]:
modified_acq_types